# FitCoach AI — Food Dataset Pipeline

Merges 3 sources into a single `food_items` schema.

| Source | File | Foods | Coverage |
|--------|------|-------|----------|
| **USDA SR Legacy** | `FoodData_Central_sr_legacy_food_csv_2018-04/` | ~7793 | International, ingredients |
| **IFCT 2017** | `ifct-2017/ifct2017.csv` | 542 | Indian whole foods |
| **Kaggle** | `kaggle-dataset/nutrients_csvfile.csv` | 335 | Common foods, macros only |

**Rules:**
- Values come ONLY from the source files — no imputation, no hallucination
- Missing values stay `NULL` (NaN in pandas)
- All numeric values are per **100g** in the output (normalised from source)
- Trace values (`t`) in Kaggle → `0.0`
- `_e` error columns in IFCT → dropped
- USDA multi-file join: `food.csv` × `food_nutrient.csv` × `nutrient.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(".")  # run from datasets/

USDA_DIR  = BASE / "FoodData_Central_sr_legacy_food_csv_2018-04"
IFCT_FILE = BASE / "ifct-2017" / "ifct2017.csv"
KAGGLE_FILE = BASE / "kaggle-dataset" / "nutrients_csvfile.csv"

for p in [USDA_DIR, IFCT_FILE, KAGGLE_FILE]:
    assert p.exists(), f"MISSING: {p}"

print("All source files found.")

All source files found.


## Output Schema

All numeric columns are **per 100g** of food.

In [2]:
OUTPUT_COLUMNS = [
    # --- identity ---
    "name",               # str  — canonical English name
    "name_normalized",    # str  — lowercase, stripped (search index)
    "aliases",            # str  — semicolon-separated regional/alt names
    # --- classification ---
    "category",           # str  — food group / category label
    "cuisine",            # str  — 'Indian' | 'Global' | ''
    # --- serving ---
    "serving_size_g",     # float — always 100.0 (all values are per 100g)
    "serving_description",# str  — human label from source e.g. '1 cup'
    # --- core macros ---
    "calories_kcal",
    "protein_g",
    "carbs_g",
    "fat_g",
    "fiber_g",
    "sugar_g",
    # --- fat breakdown ---
    "saturated_fat_g",
    "monounsaturated_fat_g",
    "polyunsaturated_fat_g",
    "trans_fat_g",
    "cholesterol_mg",
    # --- minerals ---
    "sodium_mg",
    "potassium_mg",
    "calcium_mg",
    "iron_mg",
    "magnesium_mg",
    "phosphorus_mg",
    "zinc_mg",
    # --- vitamins ---
    "vitamin_a_mcg",
    "vitamin_c_mg",
    "vitamin_d_mcg",
    "vitamin_b12_mcg",
    "folate_mcg",
    # --- diet flags ---
    "is_veg",
    "is_egg",
    "is_vegan",
]
# source + source_id are added separately via FULL_OUTPUT_COLUMNS
FULL_OUTPUT_COLUMNS = OUTPUT_COLUMNS + ["source", "source_id"]

print(f"{len(OUTPUT_COLUMNS)} nutrition columns + source traceability = {len(FULL_OUTPUT_COLUMNS)} total")

33 nutrition columns + source traceability = 35 total


---
## Source 1 — USDA SR Legacy

### Field mapping

| Output column | Source | Notes |
|---|---|---|
| name | `food.description` | |
| category | `food_category.description` | via `food.food_category_id` |
| source_id | `food.fdc_id` | |
| serving_description | `food_portion.modifier` + `gram_weight` | first portion per food |
| calories_kcal | nutrient_id=1008 | KCAL per 100g |
| protein_g | nutrient_id=1003 | G per 100g |
| carbs_g | nutrient_id=1005 | G per 100g |
| fat_g | nutrient_id=1004 | G per 100g |
| fiber_g | nutrient_id=1079 | G per 100g |
| sugar_g | nutrient_id=2000 | G per 100g |
| saturated_fat_g | nutrient_id=1257 (trans) → actually need SFA. Using closest: sum of SFA 4:0–24:0 is complex; USDA stores individual SFAs. Use nutrient_id=1258 (SFA total not stored separately) — **NOTE: USDA SR has individual SFAs, no single total SFA column. We use SFA 16:0 + 18:0 as proxy? NO — we use the NLEA total fat nutrient_id=1085 for total, and for SFA we note it's unavailable in SR as a single field.** Actually checking: nutrient 1257 = 'Fatty acids, total trans'. SFA total = not a direct field in SR legacy. We leave NULL for SR. |
| trans_fat_g | nutrient_id=1257 | |
| cholesterol_mg | nutrient_id=1253 | MG per 100g |
| sodium_mg | nutrient_id=1093 | MG per 100g |
| potassium_mg | nutrient_id=1092 | MG per 100g |
| calcium_mg | nutrient_id=1087 | MG per 100g |
| iron_mg | nutrient_id=1089 | MG per 100g |
| magnesium_mg | nutrient_id=1090 | MG per 100g |
| phosphorus_mg | nutrient_id=1091 | MG per 100g |
| zinc_mg | nutrient_id=1095 | MG per 100g |
| vitamin_a_mcg | nutrient_id=1106 (RAE) | UG per 100g |
| vitamin_c_mg | nutrient_id=1162 | MG per 100g |
| vitamin_d_mcg | nutrient_id=1114 | UG per 100g |
| vitamin_b12_mcg | nutrient_id=1178 | UG per 100g |
| folate_mcg | nutrient_id=1190 (DFE) | UG per 100g |

**Diet flags from category_id:**
- is_veg=False: categories 5 (Poultry), 7 (Sausages), 10 (Pork), 13 (Beef), 15 (Fish/Shellfish), 17 (Lamb), 21 (Fast Foods), 24 (American Indian), 25 (Restaurant), 28 (Alcoholic Beverages)
- is_egg=True: category 1 (Dairy and Egg) where 'egg' in description
- is_vegan=False: category 1 (dairy/egg), same non-veg list

In [3]:
# ── Load USDA files ─────────────────────────────────────────────────────────
food_df       = pd.read_csv(USDA_DIR / "food.csv")
food_cat_df   = pd.read_csv(USDA_DIR / "food_category.csv")
nutrient_df   = pd.read_csv(USDA_DIR / "nutrient.csv")
food_nut_df   = pd.read_csv(USDA_DIR / "food_nutrient.csv",
                            usecols=["fdc_id", "nutrient_id", "amount"])
food_por_df   = pd.read_csv(USDA_DIR / "food_portion.csv",
                            usecols=["fdc_id", "seq_num", "modifier", "gram_weight"])

print("food.csv:", len(food_df))
print("food_nutrient.csv:", len(food_nut_df))
print("nutrient.csv:", len(nutrient_df))
print("food_portion.csv:", len(food_por_df))

food.csv: 7793
food_nutrient.csv: 644125
nutrient.csv: 474
food_portion.csv: 14449


In [4]:
# ── Nutrient IDs we want ─────────────────────────────────────────────────────
USDA_NUTRIENT_MAP = {
    1008: "calories_kcal",
    1003: "protein_g",
    1005: "carbs_g",
    1004: "fat_g",
    1079: "fiber_g",
    2000: "sugar_g",
    1257: "trans_fat_g",
    1253: "cholesterol_mg",
    1093: "sodium_mg",
    1092: "potassium_mg",
    1087: "calcium_mg",
    1089: "iron_mg",
    1090: "magnesium_mg",
    1091: "phosphorus_mg",
    1095: "zinc_mg",
    1106: "vitamin_a_mcg",
    1162: "vitamin_c_mg",
    1114: "vitamin_d_mcg",
    1178: "vitamin_b12_mcg",
    1190: "folate_mcg",
}
# Verify all IDs actually exist in nutrient.csv
available_ids = set(nutrient_df["id"].astype(int))
for nid, col in USDA_NUTRIENT_MAP.items():
    status = "OK" if nid in available_ids else "MISSING"
    print(f"  {status}  nutrient_id={nid}  →  {col}")

  OK  nutrient_id=1008  →  calories_kcal
  OK  nutrient_id=1003  →  protein_g
  OK  nutrient_id=1005  →  carbs_g
  OK  nutrient_id=1004  →  fat_g
  OK  nutrient_id=1079  →  fiber_g
  OK  nutrient_id=2000  →  sugar_g
  OK  nutrient_id=1257  →  trans_fat_g
  OK  nutrient_id=1253  →  cholesterol_mg
  OK  nutrient_id=1093  →  sodium_mg
  OK  nutrient_id=1092  →  potassium_mg
  OK  nutrient_id=1087  →  calcium_mg
  OK  nutrient_id=1089  →  iron_mg
  OK  nutrient_id=1090  →  magnesium_mg
  OK  nutrient_id=1091  →  phosphorus_mg
  OK  nutrient_id=1095  →  zinc_mg
  OK  nutrient_id=1106  →  vitamin_a_mcg
  OK  nutrient_id=1162  →  vitamin_c_mg
  OK  nutrient_id=1114  →  vitamin_d_mcg
  OK  nutrient_id=1178  →  vitamin_b12_mcg
  OK  nutrient_id=1190  →  folate_mcg


In [5]:
# ── Pivot food_nutrient → wide (one row per food, one col per nutrient) ──────
wanted_ids = list(USDA_NUTRIENT_MAP.keys())
fn_filtered = food_nut_df[food_nut_df["nutrient_id"].isin(wanted_ids)].copy()
fn_filtered["col_name"] = fn_filtered["nutrient_id"].map(USDA_NUTRIENT_MAP)

# Keep one value per (fdc_id, nutrient_id) — take first if duplicates
fn_filtered = fn_filtered.drop_duplicates(subset=["fdc_id", "nutrient_id"])

nutrients_wide = fn_filtered.pivot(index="fdc_id", columns="col_name", values="amount").reset_index()
print("Pivoted nutrients shape:", nutrients_wide.shape)

Pivoted nutrients shape: (7793, 21)


In [6]:
# ── Merge food + category + nutrients ────────────────────────────────────────
food_cat_df = food_cat_df.rename(columns={"description": "category"})
usda = food_df.merge(food_cat_df[["id", "category"]], left_on="food_category_id", right_on="id", how="left")
usda = usda.merge(nutrients_wide, on="fdc_id", how="left")

print("After merge:", usda.shape)

After merge: (7793, 27)


In [7]:
# ── Best serving description per food (seq_num=1, first entry) ───────────────
best_portion = (
    food_por_df.sort_values("seq_num")
    .groupby("fdc_id").first()
    .reset_index()
)
best_portion["serving_description"] = (
    best_portion["modifier"].fillna("") + " (" +
    best_portion["gram_weight"].astype(str) + "g)"
).str.strip()

usda = usda.merge(best_portion[["fdc_id", "serving_description"]], on="fdc_id", how="left")

In [8]:
# ── Diet flags from category_id ──────────────────────────────────────────────
NON_VEG_CATS   = {5, 7, 10, 13, 15, 17}   # Poultry, Sausages, Pork, Beef, Fish, Lamb
MIXED_VEG_CATS = {21, 25}                  # Fast Foods, Restaurant — mixed, flag as non-veg
EGG_CAT        = {1}                       # Dairy and Egg
DAIRY_CAT      = {1}                       # Dairy and Egg → not vegan

cat_id = usda["food_category_id"].fillna(0).astype(int)
desc_lower = usda["description"].str.lower().fillna("")

usda["is_veg"]   = ~(cat_id.isin(NON_VEG_CATS | MIXED_VEG_CATS))
usda["is_egg"]   = (cat_id.isin(EGG_CAT)) & (desc_lower.str.contains("egg"))
usda["is_vegan"] = usda["is_veg"] & ~(cat_id.isin(DAIRY_CAT))

print("is_veg breakdown:")
print(usda["is_veg"].value_counts())

is_veg breakdown:
is_veg
True     4804
False    2989
Name: count, dtype: int64


In [9]:
# ── Assemble USDA output frame ────────────────────────────────────────────────
usda_out = pd.DataFrame()
usda_out["name"]               = usda["description"]
usda_out["name_normalized"]    = usda["description"].str.lower().str.strip()
usda_out["aliases"]            = None
usda_out["category"]           = usda["category"]
usda_out["cuisine"]            = "Global"
usda_out["source"]             = "USDA_SR"
usda_out["source_id"]          = usda["fdc_id"].astype(str)
usda_out["serving_size_g"]     = 100.0
usda_out["serving_description"] = usda["serving_description"]
# macros
for col in ["calories_kcal", "protein_g", "carbs_g", "fat_g", "fiber_g", "sugar_g",
            "trans_fat_g", "cholesterol_mg",
            "sodium_mg", "potassium_mg", "calcium_mg", "iron_mg",
            "magnesium_mg", "phosphorus_mg", "zinc_mg",
            "vitamin_a_mcg", "vitamin_c_mg", "vitamin_d_mcg",
            "vitamin_b12_mcg", "folate_mcg"]:
    usda_out[col] = usda[col] if col in usda.columns else np.nan
# these are not in SR Legacy as single columns
usda_out["saturated_fat_g"]      = np.nan
usda_out["monounsaturated_fat_g"]= np.nan
usda_out["polyunsaturated_fat_g"]= np.nan
# diet flags
usda_out["is_veg"]   = usda["is_veg"]
usda_out["is_egg"]   = usda["is_egg"]
usda_out["is_vegan"] = usda["is_vegan"]

print("USDA output shape:", usda_out.shape)
usda_out.head(3)

USDA output shape: (7793, 35)


,name,name_normalized,aliases,category,cuisine,source,source_id,serving_size_g,serving_description,calories_kcal,...,vitamin_c_mg,vitamin_d_mcg,vitamin_b12_mcg,folate_mcg,saturated_fat_g,monounsaturated_fat_g,polyunsaturated_fat_g,is_veg,is_egg,is_vegan
0,"Pillsbury Golden Layer Buttermilk Biscuits, Ar...","pillsbury golden layer buttermilk biscuits, ar...",None,Baked Products,Global,USDA_SR,167512,100.0,serving (34.0g),307.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,True
1,"Pillsbury, Cinnamon Rolls with Icing, refriger...","pillsbury, cinnamon rolls with icing, refriger...",None,Baked Products,Global,USDA_SR,167513,100.0,serving 1 roll with icing (44.0g),330.0,...,0.1,NaN,NaN,NaN,NaN,NaN,NaN,True,False,True
2,"Kraft Foods, Shake N Bake Original Recipe, Coa...","kraft foods, shake n bake original recipe, coa...",None,Baked Products,Global,USDA_SR,167514,100.0,serving (28.0g),377.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,True


---
## Source 2 — IFCT 2017

### Field mapping

All IFCT values are stored per 100g in the source.

| Output column | IFCT column | Notes |
|---|---|---|
| name | `name` | |
| aliases | `lang` | semicolon-separated regional names |
| category | `grup` | e.g. 'Cereals and Millets' |
| source_id | `code` | e.g. 'A001' |
| calories_kcal | `enerc` | kJ? NO — IFCT stores kcal directly |
| protein_g | `protcnt` | |
| carbs_g | `choavldf` | available carbs by difference |
| fat_g | `fatce` | |
| fiber_g | `fibtg` | total dietary fiber |
| sugar_g | `fsugar` | free sugars |
| saturated_fat_g | `fasat` | |
| monounsaturated_fat_g | `fams` | |
| polyunsaturated_fat_g | `fapu` | |
| trans_fat_g | `fatrn` | |
| cholesterol_mg | `cholc` | stored as g/100g → × 1000 for mg |
| sodium_mg | `na` | stored as g/100g → × 1000 |
| potassium_mg | `k` | stored as g/100g → × 1000 |
| calcium_mg | `ca` | stored as g/100g → × 1000 |
| iron_mg | `fe` | stored as g/100g → × 1000 |
| magnesium_mg | `mg` | stored as g/100g → × 1000 |
| phosphorus_mg | `p` | stored as g/100g → × 1000 |
| zinc_mg | `zn` | stored as g/100g → × 1000 |
| vitamin_a_mcg | `vita` | stored as g/100g → × 1,000,000 mcg |
| vitamin_c_mg | `vitc` | stored as g/100g → × 1000 |
| vitamin_d_mcg | `vitd` | stored as g/100g → × 1,000,000 |
| vitamin_b12_mcg | (derived from `vitb`) | not a direct field — leave NULL |
| folate_mcg | `folsum` | stored as g/100g → × 1,000,000 |

**Unit note:** IFCT stores mineral/vitamin values in **g per 100g** (very small numbers like 0.000181 for calcium).
- Minerals (Ca, Fe, Mg, P, K, Na, Zn): g/100g → mg/100g = × 1000
- Vitamins (vita, vitc, vitd, folsum): g/100g → mcg/100g = × 1,000,000

**Diet flags from `tags` column:**
- `veg` tag → is_veg=True, is_vegan=True
- `vegetarian` tag → is_veg=True, is_vegan=True (IFCT uses both synonymously)
- `eggetarian` tag (and NOT veg) → is_egg=True, is_veg=True, is_vegan=False
- `fishetarian` → is_veg=False, is_egg=False
- `nonveg` → is_veg=False, is_egg=False, is_vegan=False

In [10]:
ifct = pd.read_csv(IFCT_FILE, low_memory=False)
print("IFCT shape:", ifct.shape)
print("Columns:", list(ifct.columns[:20]), "...")

IFCT shape: (542, 421)
Columns: ['code', 'name', 'scie', 'lang', 'grup', 'regn', 'tags', 'enerc', 'enerc_e', 'water', 'water_e', 'ash', 'ash_e', 'vit', 'vit_e', 'fatce', 'fatce_e', 'cholc', 'cholc_e', 'fibtg'] ...


In [11]:
# Drop all _e (error/uncertainty) columns
error_cols = [c for c in ifct.columns if c.endswith("_e")]
ifct = ifct.drop(columns=error_cols)
print(f"Dropped {len(error_cols)} error columns. Remaining: {ifct.shape[1]}")

Dropped 207 error columns. Remaining: 214


In [12]:
# Verify the columns we're mapping actually exist
IFCT_REQUIRED_COLS = [
    "code", "name", "lang", "grup", "tags",
    "enerc", "protcnt", "choavldf", "fatce", "fibtg", "fsugar",
    "fasat", "fams", "fapu", "fatrn", "cholc",
    "na", "k", "ca", "fe", "mg", "p", "zn",
    "vita", "vitc", "vitd", "folsum",
]
for col in IFCT_REQUIRED_COLS:
    status = "OK" if col in ifct.columns else "MISSING"
    print(f"  {status}  {col}")

  OK  code
  OK  name
  OK  lang
  OK  grup
  OK  tags
  OK  enerc
  OK  protcnt
  OK  choavldf
  OK  fatce
  OK  fibtg
  OK  fsugar
  OK  fasat
  OK  fams
  OK  fapu
  OK  fatrn
  OK  cholc
  OK  na
  OK  k
  OK  ca
  OK  fe
  OK  mg
  OK  p
  OK  zn
  OK  vita
  OK  vitc
  OK  vitd
  OK  folsum


In [13]:
# Verify units: print min/max of mineral cols to confirm g/100g scale
sample_minerals = ["ca", "fe", "na", "k"]
print("Raw mineral values (should be tiny, in g/100g):")
print(ifct[sample_minerals].describe().loc[["min", "max", "mean"]])

Raw mineral values (should be tiny, in g/100g):
            ca        fe        na         k
min   0.000000  0.000000  0.000000  0.000000
max   1.664000  0.053110  0.849000  2.374000
mean  0.070914  0.002746  0.038507  0.370008


In [14]:
# Unit conversions — CONFIRMED: IFCT stores g/100g for minerals and vitamins
# Minerals: g/100g → mg/100g (× 1000)
MINERAL_G_TO_MG = ["na", "k", "ca", "fe", "mg", "p", "zn"]
# Vitamins: g/100g → mcg/100g (× 1_000_000)
VITAMIN_G_TO_MCG = ["vita", "vitc", "vitd", "folsum"]
# vitc output is mg not mcg — so: g/100g × 1000 = mg/100g
VITAMIN_G_TO_MG  = ["vitc"]

for col in MINERAL_G_TO_MG:
    ifct[col] = pd.to_numeric(ifct[col], errors="coerce") * 1000

for col in ["vita", "vitd", "folsum"]:
    ifct[col] = pd.to_numeric(ifct[col], errors="coerce") * 1_000_000

ifct["vitc"] = pd.to_numeric(ifct["vitc"], errors="coerce") * 1000

# cholesterol: g/100g → mg/100g
ifct["cholc"] = pd.to_numeric(ifct["cholc"], errors="coerce") * 1000

print("After conversion — calcium sample (mg/100g):")
print(ifct[["name","ca"]].head(5))

After conversion — calcium sample (mg/100g):
                        name      ca
0       Amaranth seed, black  181.00
1  Amaranth seed, pale brown  162.00
2                      Bajra   27.35
3                     Barley   28.64
4                      Jowar   27.60


In [15]:
# ── Diet flags from tags column ───────────────────────────────────────────────
def parse_ifct_flags(tags_str):
    tags = set(str(tags_str).lower().split())
    is_veg   = "veg" in tags or "vegetarian" in tags
    is_egg   = "eggetarian" in tags and not is_veg
    # veg foods in IFCT include eggetarian as a compatible diet — but the food itself
    # is classified as veg if grup is in plant categories
    # Simpler: use grup-level group tags
    is_veg   = "veg" in tags
    is_egg   = ("eggetarian" in tags) and ("veg" not in tags)
    is_vegan = is_veg and ("eggetarian" not in tags)
    return pd.Series({"is_veg": is_veg, "is_egg": is_egg, "is_vegan": is_vegan})

flags = ifct["tags"].apply(parse_ifct_flags)
ifct = pd.concat([ifct, flags], axis=1)

print("Diet flags distribution:")
print(ifct[["is_veg","is_egg","is_vegan"]].value_counts().head(10))

Diet flags distribution:
is_veg  is_egg  is_vegan
True    False   False       328
False   False   False       199
        True    False        15
Name: count, dtype: int64


In [16]:
# ── Assemble IFCT output frame ────────────────────────────────────────────────
ifct_out = pd.DataFrame()
ifct_out["name"]               = ifct["name"]
ifct_out["name_normalized"]    = ifct["name"].str.lower().str.strip()
ifct_out["aliases"]            = ifct["lang"]
ifct_out["category"]           = ifct["grup"]
ifct_out["cuisine"]            = "Indian"
ifct_out["source"]             = "IFCT2017"
ifct_out["source_id"]          = ifct["code"]
ifct_out["serving_size_g"]     = 100.0
ifct_out["serving_description"] = None
# IFCT enerc is in kJ — convert to kcal (÷ 4.184)
ifct_out["calories_kcal"]      = pd.to_numeric(ifct["enerc"], errors="coerce") / 4.184
ifct_out["protein_g"]          = pd.to_numeric(ifct["protcnt"], errors="coerce")
ifct_out["carbs_g"]            = pd.to_numeric(ifct["choavldf"], errors="coerce")
ifct_out["fat_g"]              = pd.to_numeric(ifct["fatce"], errors="coerce")
ifct_out["fiber_g"]            = pd.to_numeric(ifct["fibtg"], errors="coerce")
ifct_out["sugar_g"]            = pd.to_numeric(ifct["fsugar"], errors="coerce")
ifct_out["saturated_fat_g"]    = pd.to_numeric(ifct["fasat"], errors="coerce")
ifct_out["monounsaturated_fat_g"] = pd.to_numeric(ifct["fams"], errors="coerce")
ifct_out["polyunsaturated_fat_g"] = pd.to_numeric(ifct["fapu"], errors="coerce")
ifct_out["trans_fat_g"]        = pd.to_numeric(ifct["fatrn"], errors="coerce")
ifct_out["cholesterol_mg"]     = ifct["cholc"]
ifct_out["sodium_mg"]          = ifct["na"]
ifct_out["potassium_mg"]       = ifct["k"]
ifct_out["calcium_mg"]         = ifct["ca"]
ifct_out["iron_mg"]            = ifct["fe"]
ifct_out["magnesium_mg"]       = ifct["mg"]
ifct_out["phosphorus_mg"]      = ifct["p"]
ifct_out["zinc_mg"]            = ifct["zn"]
ifct_out["vitamin_a_mcg"]      = ifct["vita"]
ifct_out["vitamin_c_mg"]       = ifct["vitc"]
ifct_out["vitamin_d_mcg"]      = ifct["vitd"]
ifct_out["vitamin_b12_mcg"]    = np.nan   # no direct B12 column in IFCT
ifct_out["folate_mcg"]         = ifct["folsum"]
ifct_out["is_veg"]             = ifct["is_veg"]
ifct_out["is_egg"]             = ifct["is_egg"]
ifct_out["is_vegan"]           = ifct["is_vegan"]

print("IFCT output shape:", ifct_out.shape)
print("Calories sample (should be ~350-400 for grains):")
print(ifct_out[["name","calories_kcal"]].head(5))

IFCT output shape: (542, 35)
Calories sample (should be ~350-400 for grains):
                        name  calories_kcal
0       Amaranth seed, black     356.118547
1  Amaranth seed, pale brown     355.879541
2                      Bajra     347.992352
3                     Barley     315.726577
4                      Jowar     334.130019


---
## Source 3 — Kaggle

### Field mapping

Kaggle values are **per serving** (Grams column = serving weight). We normalise to per 100g.

| Output column | Kaggle column | Normalisation |
|---|---|---|
| name | `Food` | |
| category | `Category` | raw string (messy — see note) |
| serving_description | `Measure` | human serving label |
| calories_kcal | `Calories` | ÷ Grams × 100 |
| protein_g | `Protein` | ÷ Grams × 100 |
| fat_g | `Fat` | ÷ Grams × 100 |
| saturated_fat_g | `Sat.Fat` | ÷ Grams × 100 (trace → 0) |
| fiber_g | `Fiber` | ÷ Grams × 100 |
| carbs_g | `Carbs` | ÷ Grams × 100 |

**Not available in Kaggle:** sugar, all minerals, all vitamins, monounsaturated/polyunsaturated/trans fat, cholesterol → all NULL

**Data quality issues:**
- `t` (trace) in numeric fields → replace with `0.0`
- Grams column has commas in numbers (e.g. `1,419`) → strip commas
- Category column is messy (numbers bleeding from Carbs column) → clean to known string categories only

In [17]:
kaggle = pd.read_csv(KAGGLE_FILE)
print("Kaggle shape:", kaggle.shape)
print(kaggle.dtypes)
kaggle.head(5)

Kaggle shape: (335, 10)
Food        str
Measure     str
Grams       str
Calories    str
Protein     str
Fat         str
Sat.Fat     str
Fiber       str
Carbs       str
Category    str
dtype: object


,Food,Measure,Grams,Calories,Protein,Fat,Sat.Fat,Fiber,Carbs,Category
0,Cows' milk,1 qt.,976,660,32,40,36,0,48,Dairy products
1,Milk skim,1 qt.,984,360,36,t,t,0,52,Dairy products
2,Buttermilk,1 cup,246,127,9,5,4,0,13,Dairy products
3,"Evaporated, undiluted",1 cup,252,345,16,20,18,0,24,Dairy products
4,Fortified milk,6 cups,"1,419","1,373",89,42,23,1.4,119,Dairy products


In [18]:
# ── Clean Kaggle numeric columns ──────────────────────────────────────────────
def clean_num(series):
    """Replace 't' (trace) with 0, strip commas, cast to float."""
    return (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace("t", "0")
        .pipe(pd.to_numeric, errors="coerce")
    )

for col in ["Grams", "Calories", "Protein", "Fat", "Sat.Fat", "Fiber", "Carbs"]:
    kaggle[col] = clean_num(kaggle[col])

# Drop corrupted rows: Protein or Fat > serving Grams is physically impossible
# (e.g. Butter row where Protein=114, Grams=112 — index bleed in source CSV)
bad_rows = (
    (kaggle["Protein"] > kaggle["Grams"]) |
    (kaggle["Fat"] > kaggle["Grams"]) |
    (kaggle["Carbs"] > kaggle["Grams"])
)
print(f"Dropping {bad_rows.sum()} corrupted rows (macro > serving weight):")
print(kaggle[bad_rows][["Food","Grams","Protein","Fat","Carbs"]])
kaggle = kaggle[~bad_rows].copy()

print(f"\nAfter cleaning — rows: {len(kaggle)}")
print("Grams nulls:", kaggle["Grams"].isna().sum())
print("Grams == 0:", (kaggle["Grams"] == 0).sum())

Dropping 4 corrupted rows (macro > serving weight):
            Food  Grams  Protein    Fat  Carbs
30        Butter    112      114  115.0  118.0
31        Butter    112      114  115.0  118.0
82       Oysters    230      232  233.0  236.0
220  Bran flakes     25        3    0.0   32.0

After cleaning — rows: 331
Grams nulls: 0
Grams == 0: 0


In [19]:
# Drop rows where Grams is 0 or null (can't normalise)
kaggle = kaggle[kaggle["Grams"].notna() & (kaggle["Grams"] > 0)].copy()
print("After dropping bad Grams rows:", len(kaggle))

After dropping bad Grams rows: 331


In [20]:
# ── Normalise to per-100g ─────────────────────────────────────────────────────
factor = 100.0 / kaggle["Grams"]
for col in ["Calories", "Protein", "Fat", "Sat.Fat", "Fiber", "Carbs"]:
    kaggle[f"{col}_per100"] = kaggle[col] * factor

print("Normalised. Sample:")
print(kaggle[["Food", "Grams", "Calories", "Calories_per100"]].head(5))

Normalised. Sample:
                    Food  Grams  Calories  Calories_per100
0             Cows' milk    976     660.0        67.622951
1              Milk skim    984     360.0        36.585366
2             Buttermilk    246     127.0        51.626016
3  Evaporated, undiluted    252     345.0       136.904762
4         Fortified milk   1419    1373.0        96.758280


In [21]:
# ── Clean Category — keep only recognisable string categories ────────────────
KNOWN_CATEGORIES = {
    "Dairy products", "Vegetables A-E", "Vegetables R-Z", "Vegetables F-P",
    "Fruits A-F", "Fruits G-P", "Fruits R-Z",
    "Meat", "Fish", "Drinks", "Breads", "Desserts",
    "Fats", "Seeds and Nuts", "Soups", "Jams",
}
# Normalise: strip quotes, whitespace
kaggle["Category_clean"] = (
    kaggle["Category"].astype(str)
    .str.strip('"').str.strip()
)
# Mark rows whose category starts with a digit as messy (CSV parse overflow)
kaggle["cat_valid"] = ~kaggle["Category_clean"].str.match(r'^\d')

print("Valid category rows:", kaggle["cat_valid"].sum())
print("Dropped (numeric category — parse error):", (~kaggle["cat_valid"]).sum())

kaggle_clean = kaggle[kaggle["cat_valid"]].copy()
print("Unique clean categories:", kaggle_clean["Category_clean"].nunique())
print(kaggle_clean["Category_clean"].value_counts())

Valid category rows: 331
Dropped (numeric category — parse error): 0
Unique clean categories: 16
Category_clean
Breads, cereals, fastfood,grains    44
Meat, Poultry                       30
Desserts, sweets                    29
Dairy products                      28
Vegetables A-E                      28
Vegetables R-Z                      28
Fruits G-P                          28
Fruits A-F                          22
Fish, Seafood                       18
Vegetables F-P                      14
Fats, Oils, Shortenings             12
Seeds and Nuts                      12
Drinks,Alcohol, Beverages           12
Soups                               10
Fruits R-Z                           8
Jams, Jellies                        8
Name: count, dtype: int64


In [22]:
# ── Diet flags from Kaggle category ──────────────────────────────────────────
NON_VEG_KAG = {"Meat", "Fish"}
cat = kaggle_clean["Category_clean"]
kaggle_clean["is_veg"]   = ~cat.isin(NON_VEG_KAG)
kaggle_clean["is_egg"]   = False
# Dairy products → not vegan
kaggle_clean["is_vegan"] = kaggle_clean["is_veg"] & ~(cat == "Dairy products")

In [23]:
# ── Assemble Kaggle output frame ──────────────────────────────────────────────
kaggle_out = pd.DataFrame()
kaggle_out["name"]               = kaggle_clean["Food"]
kaggle_out["name_normalized"]    = kaggle_clean["Food"].str.lower().str.strip()
kaggle_out["aliases"]            = None
kaggle_out["category"]           = kaggle_clean["Category_clean"]
kaggle_out["cuisine"]            = "Global"
kaggle_out["source"]             = "KAGGLE"
kaggle_out["source_id"]          = kaggle_clean.index.astype(str)
kaggle_out["serving_size_g"]     = 100.0
kaggle_out["serving_description"] = kaggle_clean["Measure"]
kaggle_out["calories_kcal"]      = kaggle_clean["Calories_per100"]
kaggle_out["protein_g"]          = kaggle_clean["Protein_per100"]
kaggle_out["carbs_g"]            = kaggle_clean["Carbs_per100"]
kaggle_out["fat_g"]              = kaggle_clean["Fat_per100"]
kaggle_out["fiber_g"]            = kaggle_clean["Fiber_per100"]
kaggle_out["sugar_g"]            = np.nan  # not in Kaggle
kaggle_out["saturated_fat_g"]    = kaggle_clean["Sat.Fat_per100"]
kaggle_out["monounsaturated_fat_g"] = np.nan
kaggle_out["polyunsaturated_fat_g"] = np.nan
kaggle_out["trans_fat_g"]        = np.nan
kaggle_out["cholesterol_mg"]     = np.nan
# all micronutrients not in Kaggle
for col in ["sodium_mg","potassium_mg","calcium_mg","iron_mg","magnesium_mg",
            "phosphorus_mg","zinc_mg","vitamin_a_mcg","vitamin_c_mg",
            "vitamin_d_mcg","vitamin_b12_mcg","folate_mcg"]:
    kaggle_out[col] = np.nan
kaggle_out["is_veg"]   = kaggle_clean["is_veg"]
kaggle_out["is_egg"]   = kaggle_clean["is_egg"]
kaggle_out["is_vegan"] = kaggle_clean["is_vegan"]

print("Kaggle output shape:", kaggle_out.shape)
kaggle_out.head(3)

Kaggle output shape: (331, 35)


,name,name_normalized,aliases,category,cuisine,source,source_id,serving_size_g,serving_description,calories_kcal,...,phosphorus_mg,zinc_mg,vitamin_a_mcg,vitamin_c_mg,vitamin_d_mcg,vitamin_b12_mcg,folate_mcg,is_veg,is_egg,is_vegan
0,Cows' milk,cows' milk,None,Dairy products,Global,KAGGLE,0,100.0,1 qt.,67.622951,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,False
1,Milk skim,milk skim,None,Dairy products,Global,KAGGLE,1,100.0,1 qt.,36.585366,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,False
2,Buttermilk,buttermilk,None,Dairy products,Global,KAGGLE,2,100.0,1 cup,51.626016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,False


---
## Combine & Deduplicate

In [24]:
# ── Stack all three sources into FULL_OUTPUT_COLUMNS ─────────────────────────
combined = pd.concat(
    [usda_out, ifct_out, kaggle_out],
    ignore_index=True
)[FULL_OUTPUT_COLUMNS]

print("Combined shape (pre-dedup):", combined.shape)
print("\nSource counts:")
print(combined["source"].value_counts())

Combined shape (pre-dedup):

 (8666, 35)

Source counts:
source
USDA_SR     7793
IFCT2017     542
KAGGLE       331
Name: count, dtype: int64


In [25]:
# Combined is already built in the cell above using FULL_OUTPUT_COLUMNS
print("Final shape (pre-dedup):", combined.shape)

Final shape (pre-dedup): (8666, 35)


In [26]:
# ── Deduplication ─────────────────────────────────────────────────────────────
# Strategy: exact match on name_normalized.
# Priority: IFCT2017 > USDA_SR > KAGGLE
# (IFCT preferred for Indian foods, which are also sometimes in USDA)

SOURCE_PRIORITY = {"IFCT2017": 0, "USDA_SR": 1, "KAGGLE": 2}
combined["_priority"] = combined["source"].map(SOURCE_PRIORITY)

combined_dedup = (
    combined
    .sort_values("_priority")
    .drop_duplicates(subset=["name_normalized"], keep="first")
    .drop(columns=["_priority"])
    .reset_index(drop=True)
)

print(f"Before dedup: {len(combined)}  →  After: {len(combined_dedup)}")
print(f"Duplicates removed: {len(combined) - len(combined_dedup)}")

Before dedup: 8666  →  After: 8644
Duplicates removed: 22


---
## Validation

In [27]:
# ── 1. Schema check — all columns present ────────────────────────────────────
missing_cols = [c for c in FULL_OUTPUT_COLUMNS if c not in combined_dedup.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"
print("PASS: All output columns present")

PASS: All output columns present


In [28]:
# ── 2. No fabricated data — calories must come from source ───────────────────
null_cal = combined_dedup["calories_kcal"].isna().sum()
total = len(combined_dedup)
print(f"Rows with NULL calories: {null_cal}/{total} ({null_cal/total*100:.1f}%)")
# Show a sample of nulls
if null_cal > 0:
    print(combined_dedup[combined_dedup["calories_kcal"].isna()][["name","source"]].head(10))

Rows with NULL calories: 2/8644 (0.0%)
             name  source
8578  Frozen peas  KAGGLE
8618    Artichoke  KAGGLE


In [29]:
# ── 3. Sanity check — caloric ranges ─────────────────────────────────────────
cal_stats = combined_dedup["calories_kcal"].describe()
print("Calories per 100g stats:")
print(cal_stats)
# Flag extreme outliers
suspicious = combined_dedup[
    (combined_dedup["calories_kcal"] > 900) | 
    (combined_dedup["calories_kcal"] < 0)
][["name", "source", "calories_kcal"]]
print(f"\nSuspicious calorie values (>900 or <0): {len(suspicious)}")
if len(suspicious) > 0:
    print(suspicious.head(10))

Calories per 100g stats:
count    8642.000000
mean      213.798822
std       168.430272
min         0.000000
25%        84.000000
50%       174.000000
75%       309.000000
max       902.000000
Name: calories_kcal, dtype: float64

Suspicious calorie values (>900 or <0): 8
                                        name   source  calories_kcal
2283                        Fish oil, salmon  USDA_SR          902.0
2284  Fish oil, menhaden, fully hydrogenated  USDA_SR          902.0
2285                      Fish oil, menhaden  USDA_SR          902.0
2286                       Fish oil, herring  USDA_SR          902.0
3497                     Fish oil, cod liver  USDA_SR          902.0
3512                       Fish oil, sardine  USDA_SR          902.0
7211                                    Lard  USDA_SR          902.0
7212                        Fat, beef tallow  USDA_SR          902.0


In [30]:
# ── 4. Source distribution ────────────────────────────────────────────────────
print("Source distribution after dedup:")
print(combined_dedup["source"].value_counts())
print("\nCuisine distribution:")
print(combined_dedup["cuisine"].value_counts())

Source distribution after dedup:
source
USDA_SR     7793
IFCT2017     540
KAGGLE       311
Name: count, dtype: int64

Cuisine distribution:
cuisine
Global    8104
Indian     540
Name: count, dtype: int64


In [31]:
# ── 5. Macro sanity — protein+carbs+fat should be ≤ 100g (per 100g) ──────────
macro_sum = (
    combined_dedup[["protein_g","carbs_g","fat_g"]]
    .fillna(0).sum(axis=1)
)
bad_macro = combined_dedup[macro_sum > 105][["name","source","protein_g","carbs_g","fat_g"]]
print(f"Rows where protein+carbs+fat > 105g/100g: {len(bad_macro)}")
if len(bad_macro) > 0:
    print(bad_macro.head(10))

Rows where protein+carbs+fat > 105g/100g: 2
                name  source  protein_g     carbs_g  fat_g
8337      Cornflakes  KAGGLE   8.000000  100.000000    0.0
8357  Popcorn salted  KAGGLE  10.714286   71.428571   25.0


In [32]:
# ── 6. Diet flag sanity ───────────────────────────────────────────────────────
# Vegan foods must not be is_veg=False
bad_vegan = combined_dedup[(combined_dedup["is_vegan"] == True) & (combined_dedup["is_veg"] == False)]
print(f"Invalid: is_vegan=True but is_veg=False: {len(bad_vegan)}")
assert len(bad_vegan) == 0, "FAIL: Vegan flag inconsistency"
print("PASS: diet flag consistency")

Invalid: is_vegan=True but is_veg=False: 0
PASS: diet flag consistency


In [33]:
# ── 7. Coverage summary ───────────────────────────────────────────────────────
coverage = combined_dedup[[
    "calories_kcal","protein_g","carbs_g","fat_g","fiber_g",
    "sodium_mg","calcium_mg","iron_mg","vitamin_c_mg","vitamin_d_mcg"
]].notna().mean() * 100

print("Field coverage (% of rows with non-null values):")
for field, pct in coverage.items():
    bar = "█" * int(pct / 5)
    print(f"  {field:<25} {pct:5.1f}%  {bar}")

Field coverage (% of rows with non-null values):
  calories_kcal             100.0%  ███████████████████
  protein_g                 100.0%  ████████████████████
  carbs_g                   100.0%  ████████████████████
  fat_g                     100.0%  ███████████████████
  fiber_g                    93.5%  ██████████████████
  sodium_mg                  95.4%  ███████████████████
  calcium_mg                 95.4%  ███████████████████
  iron_mg                    95.5%  ███████████████████
  vitamin_c_mg               91.1%  ██████████████████
  vitamin_d_mcg              66.2%  █████████████


---
## Export

In [34]:
OUTPUT_CSV     = BASE / "output" / "food_items.csv"
OUTPUT_PARQUET = BASE / "output" / "food_items.parquet"

(BASE / "output").mkdir(exist_ok=True)

combined_dedup.to_csv(OUTPUT_CSV, index=False)
combined_dedup.to_parquet(OUTPUT_PARQUET, index=False)

print(f"Exported {len(combined_dedup)} rows")
print(f"  CSV:     {OUTPUT_CSV}")
print(f"  Parquet: {OUTPUT_PARQUET}")

Exported 8644 rows
  CSV:     output/food_items.csv
  Parquet: output/food_items.parquet


In [35]:
# ── Final summary card ────────────────────────────────────────────────────────
print("="*55)
print("  FOOD ITEMS DATASET — BUILD COMPLETE")
print("="*55)
print(f"  Total foods:      {len(combined_dedup):,}")
print(f"  Indian foods:     {(combined_dedup['cuisine']=='Indian').sum():,}  (IFCT2017)")
print(f"  Global foods:     {(combined_dedup['cuisine']=='Global').sum():,}  (USDA+Kaggle)")
print(f"  Veg foods:        {combined_dedup['is_veg'].sum():,}")
print(f"  Non-veg foods:    {(~combined_dedup['is_veg']).sum():,}")
print(f"  Output columns:   {len(combined_dedup.columns)}")
print("="*55)

  FOOD ITEMS DATASET — BUILD COMPLETE
  Total foods:      8,644
  Indian foods:     540  (IFCT2017)
  Global foods:     8,104  (USDA+Kaggle)
  Veg foods:        5,443
  Non-veg foods:    3,201
  Output columns:   35
